In [1]:
import pysam
import numpy as np
import pandas as pd
import glob
import os
from variants import SingleVariant, TrioVariant, SingleVariantLRS, TrioVariantLRS
from sklearn.model_selection import train_test_split

In [2]:
dataset = pd.read_csv('/ifs/data/research/projects/gelana/denovocnn_lrs/data/revio_dnm/training_dataset.tsv', sep='\t')
dataset

,chrom,pos,ref,alt,hap_counts,variant_type_x,child,number_of_haplotypes,hap1_var_reads,hap1_total_reads,...,mother_number_of_haplotypes,mother_hap1_var_reads,mother_hap1_total_reads,mother_hap2_var_reads,mother_hap2_total_reads,mother_hap_na_var_reads,mother_hap_na_total_reads,mother_hap1_vaf,mother_hap2_vaf,DNM_status
0,chr1,13252342,C,T,NaN,NaN,DNA0121,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSSIBLY_NOT_DNM
1,chr1,13253645,G,C,NaN,NaN,DNA0121,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSSIBLY_NOT_DNM
2,chr1,22578139,T,A,NaN,NaN,DNA0121,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSSIBLY_NOT_DNM
3,chr1,80330738,A,G,NaN,NaN,DNA0121,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSSIBLY_NOT_DNM
4,chr1,101553191,A,T,NaN,NaN,DNA0121,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSSIBLY_NOT_DNM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36505,chr1,214834227,A,ATTATAC,defaultdict(<function count_variants_by_haplot...,ins,DNA24-12831,2.0,8.0,11.0,...,2.0,0.0,13.0,0.0,9.0,0.0,0.0,0.0,0.000000,UNKNOWN
36506,chr14,25966217,GA,G,defaultdict(<function count_variants_by_haplot...,del,DNA24-12831,2.0,0.0,11.0,...,2.0,0.0,13.0,0.0,17.0,0.0,0.0,0.0,0.000000,UNKNOWN
36507,chr19,37293445,C,A,defaultdict(<function count_variants_by_haplot...,snp,DNA24-12831,3.0,0.0,8.0,...,3.0,0.0,37.0,6.0,22.0,0.0,18.0,0.0,0.272727,UNKNOWN
36508,chr2,232995222,T,C,defaultdict(<function count_variants_by_haplot...,snp,DNA24-12831,2.0,1.0,18.0,...,2.0,0.0,5.0,1.0,16.0,0.0,0.0,0.0,0.062500,UNKNOWN


In [3]:
# gets the id-s of all children
children = dataset['child'].values.tolist()
ids = np.unique(children)
ids

array(['DNA0121', 'DNA02-00077', 'DNA07-06065', 'DNA08-03179',
       'DNA09-06071', 'DNA09-15159', 'DNA10-12564', 'DNA11-04822',
       'DNA11-26362', 'DNA11-26445', 'DNA12-01403', 'DNA13-01861',
       'DNA13-05777', 'DNA13-05977', 'DNA13-08435', 'DNA13-11099',
       'DNA13-11484', 'DNA13-11628', 'DNA13-15626', 'DNA13-15765',
       'DNA13-16322', 'DNA14-00479', 'DNA14-19007', 'DNA14-19013',
       'DNA14-19814', 'DNA14-20628', 'DNA14-22376', 'DNA14-25790',
       'DNA14-26236', 'DNA14-32025', 'DNA14-33838', 'DNA14-36047',
       'DNA15-01061', 'DNA15-02690', 'DNA15-03061', 'DNA15-03937',
       'DNA15-06715', 'DNA15-09437', 'DNA15-11716', 'DNA15-12332',
       'DNA15-12880', 'DNA15-14076', 'DNA15-15161', 'DNA15-19905',
       'DNA16-07145', 'DNA16-09298', 'DNA16-10084', 'DNA16-11854',
       'DNA16-12636', 'DNA16-13779', 'DNA16-18715', 'DNA16-20326',
       'DNA16-22527', 'DNA17-00546', 'DNA17-02328', 'DNA17-05711',
       'DNA17-07626', 'DNA17-07798', 'DNA17-09105', 'DNA17-10057',

In [4]:
# function to access a specific .bam file based on a dna-id
def get_revio_bam_path(dna_id):

    files_lst = (

        glob.glob(f'/ifs/data/research/revio/work/{dna_id}/GRCh38*/{dna_id}*.bam') + 

        glob.glob(f'/ifs/data/research/revio/work/{dna_id}/GRCh38*/BAMs_*/{dna_id}*.bam')

    )


    if len(files_lst) == 0: raise FileNotFoundError(f"No bam files found for {dna_id}")

    if len(files_lst) > 1: files_lst = sorted(files_lst, key=os.path.getmtime)



    return files_lst[-1]

In [5]:
# split the data into train, validation and test
train_ids, remain = train_test_split(ids, test_size=0.3, random_state=42)# 70% train
val_ids, test_ids = train_test_split(remain, test_size=0.5, random_state=42)# 15% validation and 15% test

snp_train = dataset[(dataset['child'].isin(train_ids)) & (dataset['variant_type_x'] == 'snp')]
snp_val = dataset[(dataset['child'].isin(val_ids)) & (dataset['variant_type_x'] == 'snp')]
snp_test = dataset[(dataset['child'].isin(test_ids)) & (dataset['variant_type_x'] == 'snp')]

ins_train = dataset[(dataset['child'].isin(train_ids)) & (dataset['variant_type_x'] == 'ins')]
ins_val = dataset[(dataset['child'].isin(val_ids)) & (dataset['variant_type_x'] == 'ins')]
ins_test = dataset[(dataset['child'].isin(test_ids)) & (dataset['variant_type_x'] == 'ins')]

del_train = dataset[(dataset['child'].isin(train_ids)) & (dataset['variant_type_x'] == 'del')]
del_val = dataset[(dataset['child'].isin(val_ids)) & (dataset['variant_type_x'] == 'del')]
del_test = dataset[(dataset['child'].isin(test_ids)) & (dataset['variant_type_x'] == 'del')]

columns = {'train': [len(snp_train), len(ins_train), len(del_train), (len(snp_train) + len(ins_train) + len(del_train))],
           'validation': [len(snp_val), len(ins_val), len(del_val), (len(snp_val) + len(ins_val) + len(del_val))],
           'test': [len(snp_test), len(ins_test), len(del_test), (len(snp_test) + len(ins_test) + len(del_test))]}
indexes = ['substitution', 'insertion', 'deletion', 'total']

split_ids = [train_ids, val_ids, test_ids]
split_folders = ["train", "val", "test"]

dataset_table = pd.DataFrame(data=columns, index=indexes)
dataset_table

,train,validation,test
substitution,10190,2235,2289
insertion,369,71,93
deletion,531,115,117
total,11090,2421,2499


In [7]:
# Generate RGB images
for ids in range(len(split_ids)):
    variant_num = 0
    for id in split_ids[ids]:
        # get paths to the trio files
        trios = pd.read_csv("/ifs/data/research/revio/work/familyInfo.txt", sep= '\t')

        try:
            child_path = get_revio_bam_path(trios.loc[trios['child'] == id, 'child'].iloc[0])
            mother_path = get_revio_bam_path(trios.loc[trios['child'] == id, 'father'].iloc[0])
            father_path = get_revio_bam_path(trios.loc[trios['child'] == id, 'mother'].iloc[0])
        except:
            continue
        
        # separate the dataset based on the type of the variant
        variant_types_folders = ['substitution', 'insertion', 'deletion']
        snp = dataset[(dataset['child'].isin(split_ids[ids])) & (dataset['variant_type_x'] == 'snp')]
        ins = dataset[(dataset['child'].isin(split_ids[ids])) & (dataset['variant_type_x'] == 'ins')]
        dels = dataset[(dataset['child'].isin(split_ids[ids])) & (dataset['variant_type_x'] == 'del')]
        variant_types = [snp, ins, dels]
        
        for variant in range(len(variant_types)):
            # separate DNMs and unknown
            sub_dataset = variant_types[variant]
            possible_DNMs = sub_dataset[(sub_dataset['DNM_status'] == "POSSIBLY_PHASED_DNM") & (sub_dataset['child'] == id)]
            unknown = sub_dataset[(sub_dataset['DNM_status'] == "UNKNOWN") & (sub_dataset['child'] == id)]
        
            # generate DNM images
            for row in range(len(possible_DNMs)):
                variant_num += 1
                sample = possible_DNMs.iloc[row]
                
                child = SingleVariantLRS(sample['chrom'], int(sample['pos']), int(sample['pos']) + 1, child_path, pysam.FastaFile('/ifs/data/research/projects/ina/test_trio/GCA_000001405.15_GRCh38_full_plus_hs38d1_analysis_set.masked.fa'))
                father = SingleVariantLRS(sample['chrom'], int(sample['pos']), int(sample['pos']) + 1, father_path, pysam.FastaFile('/ifs/data/research/projects/ina/test_trio/GCA_000001405.15_GRCh38_full_plus_hs38d1_analysis_set.masked.fa'))
                mother = SingleVariantLRS(sample['chrom'], int(sample['pos']), int(sample['pos']) + 1, mother_path, pysam.FastaFile('/ifs/data/research/projects/ina/test_trio/GCA_000001405.15_GRCh38_full_plus_hs38d1_analysis_set.masked.fa'))

                trio = TrioVariantLRS(child, father, mother)
                TrioVariantLRS.save_image(f"/ifs/data/research/projects/ina/ina/DeNovoCNN/data_images/{variant_types_folders[variant]}/{split_folders[ids]}/DNMs/{id}_{sample['chrom']}_pos{sample['pos']}.png", np.flip(trio.image,2))
            
            # generate UNKNOWN images
            for row in range(len(unknown)):
                variant_num += 1
                sample = unknown.iloc[row]
                
                child = SingleVariantLRS(sample['chrom'], int(sample['pos']), int(sample['pos']) + 1, child_path, pysam.FastaFile('/ifs/data/research/projects/ina/test_trio/GCA_000001405.15_GRCh38_full_plus_hs38d1_analysis_set.masked.fa'))
                father = SingleVariantLRS(sample['chrom'], int(sample['pos']), int(sample['pos']) + 1, father_path, pysam.FastaFile('/ifs/data/research/projects/ina/test_trio/GCA_000001405.15_GRCh38_full_plus_hs38d1_analysis_set.masked.fa'))
                mother = SingleVariantLRS(sample['chrom'], int(sample['pos']), int(sample['pos']) + 1, mother_path, pysam.FastaFile('/ifs/data/research/projects/ina/test_trio/GCA_000001405.15_GRCh38_full_plus_hs38d1_analysis_set.masked.fa'))

                trio = TrioVariantLRS(child, father, mother)
                TrioVariantLRS.save_image(f"/ifs/data/research/projects/ina/ina/DeNovoCNN/data_images/{variant_types_folders[variant]}/{split_folders[ids]}/UNKNOWN/{id}_{sample['chrom']}_pos{sample['pos']}.png", np.flip(trio.image,2))
    
    print(split_folders[ids] + str(variant_num))
        

/ifs/data/research/projects/ina/ina/DeNovoCNN/code/DeNovoCNN/denovonet/variants.py:289: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  np.array(list(seq[base_start_position:base_end_position])) ==


train11090
val2421
test2499
